In [ ]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

df = pd.read_csv("../../tmp_data/merged_part_2.csv")

df.head()

,id,ceiling_height,metro_minutes,metro_walking,total_area,living_area,kitchen_area,price,utilities_amount,utilities_included,prepayment_months,is_long_rental_term,rooms_count,house_type
0,271271157,3.0,9,1,200.0,20.0,NaN,500000.0,0.0,1,1.0,1,4.0,9.0
1,271634126,3.5,8,1,198.0,95.0,18.0,500000.0,0.0,1,1.0,1,4.0,8.0
2,271173086,3.2,7,1,200.0,116.0,4.0,500000.0,0.0,1,1.0,1,4.0,NaN
3,272197456,3.2,3,1,170.0,95.0,17.0,400000.0,0.0,1,1.0,1,4.0,NaN
4,273614615,3.9,7,1,58.0,38.0,5.0,225000.0,0.0,1,1.0,1,2.0,0.0


In [2]:
# Площадь

# Процент пропусков
print("Процент пропусков в living_area: ", df["living_area"].isna().mean() * 100)
print("Процент пропусков в kitchen_area: ", df["kitchen_area"].isna().mean() * 100)

# Флаги до фильтрации
df["living_area_known"] = df["living_area"].notna().astype(int)
df["kitchen_area_known"] = df["kitchen_area"].notna().astype(int)

# Заполняем медианой по rooms_count + house_type
for col in ["living_area", "kitchen_area"]:
    df[col] = df.groupby(["rooms_count", "house_type"])[col].transform(
        lambda x: x.fillna(x.median())
    )
    # Fallback на только rooms_count
    df[col] = df.groupby("rooms_count")[col].transform(lambda x: x.fillna(x.median()))
    # Fallback на глобальную медиану
    df[col] = df[col].fillna(df[col].median())

# Площадь — фильтруем после заполнения
df = df[df["living_area"] < df["total_area"]]
df = df[df["kitchen_area"] < df["total_area"]]

df[["living_area", "kitchen_area"]].isna().sum()

Процент пропусков в living_area:  12.717223082025205
Процент пропусков в kitchen_area:  22.72385584788857


living_area     0
kitchen_area    0
dtype: int64

In [ ]:
df.to_csv("../../tmp_data/bek_processed_part_3.csv", index=False)
df.head()